<a href="https://colab.research.google.com/github/qudwo9969-glitch/maritime-data-mining/blob/main/%ED%95%B4%EC%82%AC%EB%8D%B0%EC%9D%B4%ED%84%B0%EB%A7%88%EC%9D%B4%EB%8B%9D%2011%EC%A3%BC%EC%B0%A8%20%EA%B3%BC%EC%A0%9C%20(%EC%B9%B4%EC%9D%B4%EC%A0%9C%EA%B3%B1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# 0-1. 구글 드라이브 마운트
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# 드라이브 연결
drive.mount('/content/drive')

# CSV가 저장된 폴더 경로 설정
# 예시 폴더: MyDrive/type3_week11
# 'gender_preference.csv' 파일 자체가 아니라, 파일들이 있는 디렉토리를 지정해야 합니다.
# 커널 상태에 따르면, 파일들은 '/content/'에 직접 있습니다.
DATA_DIR = Path('/content/')

print('설정된 데이터 폴더:', DATA_DIR)

# 0-2. 파일 존재 여부 확인
required_files = ['channel_purchase.csv', 'gender_preference.csv']

print('[파일 존재 여부 확인]')
for fname in required_files:
    fpath = DATA_DIR / fname
    print(f'{fname}:', 'OK' if fpath.exists() else '없음')

# 1-4. 데이터 불러오기
file_path = DATA_DIR / 'channel_purchase.csv'
df = pd.read_csv(file_path)

print('[데이터 상위 5행]')
print(df.head())
print('\n[기본 정보]')
print(df.info())

# 1-5. 교차표 생성
ct = pd.crosstab(df['channel'], df['purchase_yn'])

print('[교차표]')
print(ct)

# 1-6. 카이제곱 독립성 검정
chi2, p, dof, expected = chi2_contingency(ct)
expected_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)

print('[검정 결과]')
print('chi-square statistic:', round(chi2, 4))
print('p-value:', round(p, 6))
print('degrees of freedom:', dof)

print('\n[기대도수]')
print(expected_df)

# 1-7. 채널별 구매전환율 확인
conversion_rate = pd.crosstab(df['channel'], df['purchase_yn'], normalize='index') * 100

print('[채널별 구매 비율(%)]')
print(conversion_rate.round(2))

print('\n[최종 해석 출력]')
if p < 0.05:
    print('p-value가 0.05보다 작으므로, 유입채널과 구매여부는 독립이 아니며 서로 관련이 있다고 해석할 수 있습니다.')
else:
    print('p-value가 0.05 이상이므로, 유입채널과 구매여부의 관련성을 확인하기 어렵습니다.')

best_channel = conversion_rate[1].idxmax() if 1 in conversion_rate.columns else conversion_rate.iloc[:, -1].idxmax()
best_rate = conversion_rate[1].max() if 1 in conversion_rate.columns else conversion_rate.iloc[:, -1].max()
print(f'구매 비율이 가장 높은 채널은 {best_channel}이며, 구매 비율은 {best_rate:.2f}%입니다.')

# 2-4. 데이터 불러오기
file_path = DATA_DIR / 'gender_preference.csv'
df = pd.read_csv(file_path)

print('[데이터 상위 5행]')
print(df.head())
print('\n[기본 정보]')
print(df.info())

# 2-5. 교차표 생성
ct = pd.crosstab(df['gender'], df['content_type'])

print('[교차표]')
print(ct)

# 2-6. 카이제곱 동질성 검정
chi2, p, dof, expected = chi2_contingency(ct)
expected_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)

print('[검정 결과]')
print('chi-square statistic:', round(chi2, 4))
print('p-value:', round(p, 6))
print('degrees of freedom:', dof)

print('\n[기대도수]')
print(expected_df)

# 2-7. 성별 내 비율과 최다 선호 콘텐츠 확인
ratio = pd.crosstab(df['gender'], df['content_type'], normalize='index') * 100
top_pref = ct.idxmax(axis=1)

print('[성별 내 콘텐츠 선호 비율(%)]')
print(ratio.round(2))

print('\n[성별별 최다 선호 콘텐츠]')
print(top_pref)

print('\n[최종 해석 출력]')
if p < 0.05:
    print('p-value가 0.05보다 작으므로, 성별에 따라 콘텐츠 선호 분포가 다르다고 해석할 수 있습니다.')
else:
    print('p-value가 0.05 이상이므로, 성별에 따른 콘텐츠 선호 분포 차이를 확인하기 어렵습니다.')

for g in top_pref.index:
    print(f'{g}의 최다 선호 콘텐츠는 {top_pref[g]}입니다.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
설정된 데이터 폴더: /content
[파일 존재 여부 확인]
channel_purchase.csv: OK
gender_preference.csv: OK
[데이터 상위 5행]
  channel  purchase_yn
0    검색광고            1
1    검색광고            1
2    검색광고            1
3    검색광고            1
4    검색광고            1

[기본 정보]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 530 entries, 0 to 529
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   channel      530 non-null    object
 1   purchase_yn  530 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 8.4+ KB
None
[교차표]
purchase_yn   0    1
channel             
SNS          55   70
검색광고         50   90
이메일          80   45
직접방문         30  110
[검정 결과]
chi-square statistic: 51.716
p-value: 0.0
degrees of freedom: 3

[기대도수]
purchase_yn          0          1
channel                          
SNS          50.707547 